In [2]:
import os
import pandas as pd
import yaml

from openai import OpenAI
from pathlib import Path
from dataclasses import dataclass

In [3]:
pwd

'c:\\Users\\RSR\\PYTHON\\ServicenowGamification\\Research'

In [4]:
@dataclass
class LLMConfig:
    root_dir : Path
    provider: str
    model: str
    base_url : Path
    temperature: float
    top_p : float
    max_tokens: int
    timeout : int 
    prediction_file: Path
    output_path: Path
    response_file: Path
    system_prompt  : str

    prompt_output_path: str
    prompt_output_file: str
    include_similar_incidents: bool
    max_similar_incidents: int
    include_gamification: bool
    include_sla_prediction: bool
    include_workload_prediction: bool
    include_agent_performance: bool
    include_manager_insights: bool
    include_knowledge_suggestion: bool
    max_prompt_length: int
    prompt_version: str    


In [5]:
os.chdir('c:\\Users\\RSR\\PYTHON\\ServicenowGamification\\src')

In [6]:
from ServicenowGamification.Utils.Common import *
from ServicenowGamification.Constant import * 
from ServicenowGamification.Entity.Config_Entity import * 
import ollama
from ollama import Client

In [7]:
os.chdir('c:\\Users\\RSR\\PYTHON\\ServicenowGamification')

In [8]:
class ConfigurationManager:
    def __init__(self,
                 CONFIG_PATH=CONFIG_FILE_PATH,
                 PARAMS_PATH=PARAMS_FILE_PATH

                 ):
         print(CONFIG_PATH)
         #ROOT_DIR = ROOT_DIR = Path.cwd()
                 # Read configuration files
         #print("ROOT_DIR :", ROOT_DIR)
         
         #config_path = ROOT_DIR / "Config" / "Config.yaml"
         #params_path = ROOT_DIR / "Params.yaml"

        
         self.config=Read_Yaml(Path("Config/Config.yaml"))
         print(f"path config yaml {Path(CONFIG_PATH)}")
         #self.config=Read_Yaml(Path(CONFIG_PATH))
         self.params=Read_Yaml(Path(PARAMS_PATH))
         self.config = Read_Yaml(Path(CONFIG_FILE_PATH))

         print(self.config)
         print(self.config.keys())

        
         self.LLM_Connect = self.config.LLM_Connect
    
    def get_llm_config(self)->LLMConfig:
               
        Config=self.LLM_Connect

        create_directories([Config.root_dir])
    
        llm_config=LLMConfig(
            root_dir=Config.root_dir,
            provider=Config.provider,
            model=Config.model, 
            base_url=Config.base_url, 
            temperature =Config.temperature,
            top_p =Config.top_p,
            max_tokens =Config.max_tokens, 
            timeout =Config.timeout, 
            prediction_file =Config.prediction_file, 
            output_path =Config.output_path,                  
            response_file =Config.response_file,
            system_prompt=Config.system_prompt,
            prompt_output_path=Config.prompt_output_path,
            prompt_output_file=Config.prompt_output_file,
            include_similar_incidents=Config.include_similar_incidents,
            max_similar_incidents=Config.max_similar_incidents,
            include_gamification=Config.include_gamification,
            include_sla_prediction=Config.include_sla_prediction,
            include_workload_prediction=Config.include_workload_prediction,
            include_agent_performance=Config.include_agent_performance,
            include_manager_insights=Config.include_manager_insights,
            include_knowledge_suggestion=Config.include_knowledge_suggestion,
            max_prompt_length=Config.max_prompt_length,
            prompt_version=Config.prompt_version
           
            )
        return llm_config


In [ ]:
import os
import pandas as pd

from ollama import Client

from ServicenowGamification.Entity.Config_Entity import LLMConfig


class ConnectLLM:

    ###############################################################
    # Constructor
    ###############################################################
    def __init__(self, config: LLMConfig):

        self.config = config
        self.client = None

    ###############################################################
    # Connect to Ollama
    ###############################################################
    def connect(self):

        try:

            print("=" * 70)
            print("Connecting to Ollama Server")
            print("=" * 70)

            print("Host  :", self.config.base_url)
            print("Model :", self.config.model)

            self.client = Client(
                host=self.config.base_url
            )

            print("Ollama Client Created Successfully")

            return self.client

        except Exception as e:

            print("Ollama Connection Error:", e)

            raise Exception(
                "Unable to create Ollama Client."
            )

    ###############################################################
    # Test Connection
    ###############################################################
    def test_connection(self):

        try:

            print("\nChecking Ollama Server...")

            models = self.client.list()

            print("Connection Successful")

            print("\nAvailable Models:")

            for model in models.models:

                print(" -", model.model)

            return True

        except Exception as e:

            print("Actual Error:")
            print(type(e))
            print(e)

            raise Exception(
                "Unable to connect to Ollama Server."
            )

    ###############################################################
    # List Models
    ###############################################################
    def list_models(self):

        try:

            models = self.client.list()

            print("=" * 70)
            print("Available Models")
            print("=" * 70)

            for model in models.models:

                print(model.model)

            return models

        except Exception as e:

            print("Model List Error:", e)

            raise e

    ###############################################################
    # Generate Response
    ###############################################################
    def generate_response(self, prompt):

        try:

            response = self.client.chat(

                model=self.config.model,

                messages=[

                    {
                        "role": "system",
                        "content": self.config.system_prompt
                    },

                    {
                        "role": "user",
                        "content": prompt
                    }

                ],

                options={

                    "temperature":
                        self.config.temperature,

                    "top_p":
                        self.config.top_p,

                    "num_predict":
                        self.config.max_tokens

                }
            )

            answer = response.message.content

            return answer

        except Exception as e:

            print("LLM Generation Error:")
            print(e)

            raise Exception(
                "Unable to Generate Response."
            )

    ###############################################################
    # Build Prompt
    ###############################################################
    def build_prompt(
        self,
        incident,
        similar_incidents=None
    ):

        prompt = f"""

You are an experienced IT Service Desk AI Assistant.

Analyze the following ServiceNow incident and provide
professional recommendations.

====================================================
CURRENT INCIDENT
====================================================

Incident Number:
{incident.get("number", "")}

Short Description:
{incident.get("short_description", "")}

Category:
{incident.get("category", "")}

Sub Category:
{incident.get("subcategory", "")}

Priority:
{incident.get("priority", "")}

Assignment Group:
{incident.get("assignment_group", "")}

Assigned To:
{incident.get("assigned_to", "")}

Resolution Hours:
{incident.get("resolution_hours", "")}

MTTR:
{incident.get("mttr", "")}

SLA Percentage:
{incident.get("sla_percentage", "")}

SLA Prediction:
{incident.get("sla_status", "")}

Workload Prediction:
{incident.get("workload_level", "")}

Agent Performance:
{incident.get("performance_prediction", "")}

Performance Confidence:
{incident.get("confidence", "")}

====================================================
GAMIFICATION
====================================================

Points:
{incident.get("points", "")}

Badges:
{incident.get("badges", "")}

Rewards:
{incident.get("rewards", "")}

Achievements:
{incident.get("achievements", "")}

Level:
{incident.get("level", "")}

Leaderboard Rank:
{incident.get("leaderboard_rank", "")}

Monthly Rank:
{incident.get("monthly_rank", "")}

====================================================
SIMILAR INCIDENTS
====================================================
"""

        # ---------------------------------------------------------
        # Add FAISS similar incidents
        # ---------------------------------------------------------

        if similar_incidents:

            for i, row in enumerate(
                similar_incidents,
                start=1
            ):

                # Handle pandas Series/DataFrame row
                if hasattr(row, "to_dict"):
                    row = row.to_dict()

                prompt += f"""

Similar Incident {i}

Incident Number:
{row.get("number", "")}

Short Description:
{row.get("short_description", "")}

Category:
{row.get("category", "")}

Priority:
{row.get("priority", "")}

Resolution:
{row.get("resolution_hours", "")} Hours

Performance:
{row.get("performance_category", "")}

"""

        else:

            prompt += """

No similar incidents were found in the knowledge base.

"""

        # ---------------------------------------------------------
        # Final Instructions
        # ---------------------------------------------------------

        prompt += """

====================================================

Based on the above information provide:

1. Executive Summary

2. Root Cause Analysis

3. SLA Risk Analysis

4. Workload Analysis

5. Agent Performance Analysis

6. Recommended Resolution Steps

7. Preventive Actions

8. Gamification Recommendation

9. Manager Recommendation

10. Knowledge Article Suggestion

Respond professionally using concise bullet points.

Do not invent information that is not present
in the incident data or similar incidents.

"""

        return prompt

    ###############################################################
    # Generate Prompts
    ###############################################################
    def generate_prompts(
        self,
        faiss_results
    ):

        print("=" * 70)
        print("Generating Prompts")
        print("=" * 70)

        # ---------------------------------------------------------
        # Load prediction file
        # ---------------------------------------------------------

        dataframe = pd.read_csv(
            self.config.prediction_file
        )

        print(
            "Prediction Data Shape:",
            dataframe.shape
        )

        print(
            "Total Incidents:",
            len(dataframe)
        )

        prompts = []

        # ---------------------------------------------------------
        # Generate prompt for every incident
        # ---------------------------------------------------------

        for idx, row in dataframe.iterrows():

            print(
                f"Generating Prompt "
                f"{idx + 1}/{len(dataframe)}"
            )

            incident = row.to_dict()

            # Get similar incidents for this incident
            similar = faiss_results.get(
                idx,
                []
            )

            prompt = self.build_prompt(
                incident,
                similar
            )

            prompts.append(prompt)

        # ---------------------------------------------------------
        # Add prompt column
        # ---------------------------------------------------------

        dataframe["prompt"] = prompts

        print("\nPrompt Generation Completed")

        return dataframe

    ###############################################################
    # Generate Recommendations
    ###############################################################
    def generate_recommendations(
        self,
        dataframe
    ):

        print("=" * 70)
        print("Generating LLM Recommendations")
        print("=" * 70)

        recommendations = []

        total = len(dataframe)

        for idx, row in dataframe.iterrows():

            print(
                f"Processing Incident "
                f"{idx + 1}/{total}"
            )

            prompt = row["prompt"]

            answer = self.generate_response(
                prompt
            )

            recommendations.append(
                answer
            )

        # ---------------------------------------------------------
        # Add recommendation column
        # ---------------------------------------------------------

        dataframe[
            "llm_recommendation"
        ] = recommendations

        # ---------------------------------------------------------
        # Save output
        # ---------------------------------------------------------

        dataframe.to_csv(
            self.config.response_file,
            index=False
        )

        print("\nLLM Recommendations Saved")

        print(
            self.config.response_file
        )

        return dataframe

In [17]:
import sys

log_file = open("debug_output.txt", "w")
sys.stdout = log_file

try:

    config = ConfigurationManager()

    llm_config = config.get_llm_config()

    llm = ConnectLLM(llm_config)

    # 1. Connect to Ollama
    llm.connect()

    # 2. Test connection
    llm.test_connection()

    # 3. List available models
    llm.list_models()

    # 4. Test simple LLM response
    response = llm.generate_response(
        "Explain SLA Compliance in ServiceNow."
    )

    print("\nLLM RESPONSE:")
    print(response)

finally:

    sys.stdout = sys.__stdout__
    log_file.close()

print("Debug output saved.")

In [ ]:
incident = {
    "number": "INC0000004",
    "short_description": "Forgot email password",
    "category": "Inquiry / Help",
    "subcategory": "Outlook",
    "priority": '1 - Critical',
    "assignment_group": "Cloud",
    "assigned_to": "Chaitanya",
    "resolution_hours": 2037.45055555555,
    "mttr": 881.998833333333,
    "sla_percentage": 20,
    "sla_status": "Breached",
    "workload_level": "Low",
    "performance_prediction": "Poor",
    "confidence": 0.98,
    "points": 450,
    "badges": "SLA Champion",
    "rewards": "Eligible",
    "achievements": "First Call Resolution",
    "level": 4,
    "leaderboard_rank": 2,
    "monthly_rank": 3
}

In [22]:
import sys

log_file = open("debug_output.txt", "w")
sys.stdout = log_file
prompt = llm.build_prompt(
    incident=incident,
    similar_incidents=[]
)

print(prompt)
sys.stdout = sys.__stdout__
log_file.close()